# U2T01 — Hugging Face Deployment

This notebook provides the deployment workflow for the final BERT models developed in the **U2T01: Adapting BERT for NLP Tasks** project.

Project standard base model:

`google-bert/bert-base-cased`

The notebook reuses the deployment scripts maintained in the project repository.

## Workflow

1. Clone/update the project repository.
2. Install deployment dependencies.
3. Authenticate securely with Hugging Face.
4. Configure the local model path and target repository.
5. Validate the model artifacts.
6. Publish the model.
7. Verify the published model from the Hugging Face Hub.

> Never store Hugging Face tokens directly inside this notebook.


## 1. Repository setup

During development we use the `damian-huggingface` branch.

After the deployment tools are merged into `main`, change `BRANCH` to `main`.


In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/RusselKu/Bert_NLP_TaskAdapt.git"
BRANCH = "damian-huggingface"

PROJECT_DIR = Path("/content/Bert_NLP_TaskAdapt")

if PROJECT_DIR.exists():
    print("Repository already exists. Updating...")
    subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "fetch", "origin"],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "checkout", BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "pull"],
        check=True,
    )
else:
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            BRANCH,
            REPO_URL,
            str(PROJECT_DIR),
        ],
        check=True,
    )

os.chdir(PROJECT_DIR)

print("Project directory:", Path.cwd())
print("Branch:", BRANCH)


## 2. Install deployment dependencies

This installs the dependencies defined by the repository instead of duplicating package lists inside the notebook.


In [ ]:
%pip install -q -r requirements-deployment.txt

print("Deployment dependencies installed.")


## 3. Hugging Face authentication

### Recommended in Google Colab

Create a Colab secret named:

`HF_TOKEN`

and enable notebook access to it.

If the secret is unavailable, the notebook falls back to the Hugging Face interactive login.

**Never paste the token into a notebook cell that will be committed to GitHub.**


In [ ]:
from huggingface_hub import login

authenticated = False

try:
    from google.colab import userdata

    token = userdata.get("HF_TOKEN")

    if token:
        login(token=token)
        authenticated = True
        print("Authenticated using the Colab HF_TOKEN secret.")

except Exception:
    pass

if not authenticated:
    from huggingface_hub import notebook_login

    print("HF_TOKEN secret not available.")
    print("Opening interactive Hugging Face login...")
    notebook_login()


## 4. Verify Hugging Face identity


In [ ]:
from huggingface_hub import HfApi

api = HfApi()
user = api.whoami()

print("Authenticated Hugging Face user:", user["name"])


## 5. Configure deployment

For the smoke test we use the small artificial BERT model generated by the project.

For a final team model, replace `MODEL_PATH` and `REPO_ID`.

Example final repository names:

- `bert-agnews-topic-classification`
- `bert-conll2003-ner`
- `bert-ud-ewt-pos`
- `bert-squad-extractive-qa`


In [ ]:
HF_USERNAME = api.whoami()["name"]

MODEL_PATH = "models/uploader-smoke-test"
REPO_ID = f"{HF_USERNAME}/bert-uploader-smoke-test"

PRIVATE_REPO = True

print("Model path:", MODEL_PATH)
print("Repository:", REPO_ID)
print("Private:", PRIVATE_REPO)


## 6. Create smoke-test model

This step is only for testing the deployment infrastructure.

**Do not use this model as an assignment result.**


In [ ]:
!python scripts/create_test_model.py


## 7. Local validation

The uploader first verifies:

- `config.json`
- BERT architecture
- model weights
- tokenizer artifacts
- Model Card


In [ ]:
!python scripts/publish_model.py \
    --model-path "{MODEL_PATH}" \
    --repo-id "{REPO_ID}" \
    --validate-only


## 8. Publication control

Publication is disabled by default.

Change:

`DO_UPLOAD = False`

to:

`DO_UPLOAD = True`

only when the model and repository configuration have been reviewed.


In [ ]:
DO_UPLOAD = False

if DO_UPLOAD:
    command = [
        "python",
        "scripts/publish_model.py",
        "--model-path",
        MODEL_PATH,
        "--repo-id",
        REPO_ID,
    ]

    if PRIVATE_REPO:
        command.append("--private")

    subprocess.run(command, check=True)

else:
    print("Upload disabled.")
    print("Set DO_UPLOAD = True after reviewing the configuration.")


## 9. Verify published model

Run this cell after the model has been uploaded.

The verification script checks the repository, tokenizer, BERT configuration, and task-specific head.


In [ ]:
DO_REMOTE_VERIFY = False

if DO_REMOTE_VERIFY:
    subprocess.run(
        [
            "python",
            "scripts/verify_hub_model.py",
            "--repo-id",
            REPO_ID,
        ],
        check=True,
    )
else:
    print("Remote verification disabled.")
    print("Set DO_REMOTE_VERIFY = True after publication.")


## Final-model checklist

Before publishing any of the four final models:

- [ ] Model uses the agreed BERT-base configuration.
- [ ] Model was exported with `save_pretrained()`.
- [ ] Matching tokenizer was exported with `save_pretrained()`.
- [ ] `config.json` is present.
- [ ] Model weights are present.
- [ ] Tokenizer files are present.
- [ ] `README.md` Model Card is complete.
- [ ] Training dataset is documented.
- [ ] Metrics are documented.
- [ ] Adaptation method is documented.
- [ ] Intended use is documented.
- [ ] Limitations are documented.
- [ ] References are documented.
- [ ] Local validation passes.
- [ ] Hugging Face publication succeeds.
- [ ] Remote verification passes.
